In [1]:
!pip install torch
!pip install transformers
!pip install peft
!pip install datasets
!pip install -U bitsandbytes
!pip install accelerate
!pip install tqdm
!pip install evaluate
!pip install --upgrade gupload

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 7.6 MB/s eta 0:00:00
   ━



---
Требуется train_ds_good.csv




In [2]:
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AdamW, get_scheduler, DataCollatorForLanguageModeling
import torch
import random
from datasets import load_dataset, Dataset
from peft import get_peft_model, prepare_model_for_kbit_training, LoraConfig
from huggingface_hub import login
import gc
from tqdm.notebook import tqdm
import evaluate
import re
import pandas as pd
import ast
import pandas as pd
from google.colab import drive
import numpy as np

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
TOKEN = ""
LORA_DIMENSION_RANK = 32
LORA_ALPHA = 16
LORA_MODULES = ["q_proj", "v_proj"]
QUANT_TYPE="bf16"
BATCH_SIZE=4
NP_SAVE_PATH='pretrained_tensors.npz'

In [5]:
class Config:
  def __init__(self):
    lora_dimension_rank = LORA_DIMENSION_RANK #из оригинала
    alpha_parameter_scaling = LORA_ALPHA
    self.peft_config = LoraConfig(lora_alpha=LORA_ALPHA, inference_mode=False, r=LORA_DIMENSION_RANK, bias = "none", task_type="CAUSAL_LM", target_modules=LORA_MODULES)
    self.bits_and_bytes_config = BitsAndBytesConfig(load_in_16bit=True,
                                 bnb_16bit_quant_type=QUANT_TYPE,
                                 bnb_16bit_compute_dtype=torch.float16,
                                 bnb_16bit_use_double_quant=True) #в оригинале используем квантизацию в 16, nf
config = Config()



Unused kwargs: ['load_in_16bit', 'bnb_16bit_quant_type', 'bnb_16bit_compute_dtype', 'bnb_16bit_use_double_quant']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


In [6]:
model_name = "meta-llama/Llama-2-7b-hf"
login(token=TOKEN)# -2

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name,ignore_mismatched_sizes=False,
                                          quantization_config=config.bits_and_bytes_config,
                                          device_map="auto"
                                          )

tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [8]:
pretrained_model = AutoModelForCausalLM.from_pretrained(model_name,ignore_mismatched_sizes=False,
                                                       quantization_config=config.bits_and_bytes_config,
                                                        output_hidden_states=False
                                                       )
pretrained_model = prepare_model_for_kbit_training(pretrained_model)
pretrained_model = get_peft_model(pretrained_model, config.peft_config)

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

In [9]:
batch_size = BATCH_SIZE

In [10]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
# Whether or not to use masked language modeling. If set to False, the labels are the same as the inputs with the padding tokens ignored(-100)
#Кажется, мы так и хотим, только чистый инференс

In [11]:
good_df_tr = pd.read_csv('train_ds_good.csv', sep='|')
good_df_tr['input_ids'] = good_df_tr['input_ids'].apply(lambda x: ast.literal_eval(x))
good_df_tr['attention_mask'] = good_df_tr['attention_mask'].apply(lambda x: ast.literal_eval(x))
good_train_ds = Dataset.from_pandas(good_df_tr)

In [12]:
good_dataloader_pr_mod = torch.utils.data.DataLoader(
        good_train_ds, batch_size=batch_size, pin_memory=True, num_workers=1, shuffle=False, collate_fn=data_collator
    )
gc.collect()

253

In [13]:
pretrained_model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear4bit(in_featur

In [14]:
result_r = []
for _, batch in tqdm(enumerate(good_dataloader_pr_mod),  total=len(good_dataloader_pr_mod),
                                                                           desc ="pretrained_outputs"):


  input_ids = torch.transpose(torch.cat(list(map(lambda el: el.unsqueeze(1), batch["input_ids"])), dim=1),0,1).to(device)
  attention_mask = torch.transpose(torch.cat(list(map(lambda el: el.unsqueeze(1), batch["attention_mask"])), dim=1),0,1).to(device)

  with torch.no_grad():
    out_logits = pretrained_model(input_ids, attention_mask = attention_mask).logits
    sf = torch.nn.functional.softmax(out_logits, dim=-1)
    prob_p = sf.chunk(sf.size(0), dim=0)
    prob_p = list(map(lambda el: el.squeeze(0),prob_p))

  for i in range(len(prob_p)):
    result_r.append(prob_p[i].detach().cpu().numpy())

pretrained_outputs:   0%|          | 0/19 [00:00<?, ?it/s]

In [15]:
np.savez(NP_SAVE_PATH, result_r)
#6 минут на пределе памяти - это работает, не эффективно, но стабильно, pickle показывает еще более худшие результаты

In [16]:
del result_r

In [17]:
gc.collect()

173

In [18]:
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
!cp $NP_SAVE_PATH /content/drive/MyDrive

Сохранение файла на диск
#################################################